# Goal: statistical-summary 

In [1]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\n'

In [2]:
import polars as pl

In [3]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-10\data\lev-10_merged.parquet"
pdf = pl.scan_parquet(path)

In [4]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('Item_Code_12_series', String),
        ('OOHP_Quantity_12_series', Float64),
        ('OOHP_Value_12_series', Float64),
        ('Total_Consumption_Quantity_12_se', Float64),
        ('Total_Consumption_Value_12_serie', Int64),
        ('Source_12_series', String),
        ('Multiplier', Int64)])

# Useful Variables

In [5]:
cols = [
'Item_Code_12_series',
'OOHP_Quantity_12_series',
'OOHP_Value_12_series',
'Total_Consumption_Quantity_12_se',
'Total_Consumption_Value_12_serie',
]

In [6]:
df = pdf.select(cols)

In [7]:
df.head(2).collect()

Item_Code_12_series,OOHP_Quantity_12_series,OOHP_Value_12_series,Total_Consumption_Quantity_12_se,Total_Consumption_Value_12_serie
str,f64,f64,f64,i64
"""322""",null,null,0.0,80
"""323""",null,null,0.0,220


In [8]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in cols]
)

In [9]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

Item_Code_12_series,OOHP_Quantity_12_series,OOHP_Value_12_series,Total_Consumption_Quantity_12_se,Total_Consumption_Value_12_serie
u32,u32,u32,u32,u32
20,71,198,313,882


# Logic

In [10]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_11780\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [11]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
Item_Code_12_series,1658620.0,0.0,314.747675,8.51939,300.0,309.0,316.0,319.0,329.0
OOHP_Quantity_12_series,5430.0,1653190.0,27.005525,26.814432,0.0,15.0,22.0,32.0,1000.0
OOHP_Value_12_series,17178.0,1641442.0,49.578531,50.98318,0.0,15.0,30.0,60.0,650.0
Total_Consumption_Quantity_12_se,898492.0,760128.0,33.881519,47.395669,0.0,3.0,16.0,50.0,2000.0
Total_Consumption_Value_12_serie,1658620.0,0.0,116.459951,141.12515,1.0,30.0,70.0,150.0,10000.0


# Categorical Columns

In [12]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))